<a href="https://colab.research.google.com/github/SSK-KGP/ReLeaf/blob/main/Plant_disease_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mount to drive

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
BASE_PATH = '/content/drive/MyDrive/PlantVillage/'
os.makedirs(BASE_PATH, exist_ok=True)

Setting up Hyperparameters

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 16
NUM_EPOCHS = 30
SEED = 42
MODEL_SAVE = BASE_PATH + "best_plant_disease_cnn.keras"
LABELS_SAVE = BASE_PATH + "class_names.json"
DATA_SUBDIR = "color"
DATASET_NAME = "abdallahalidev/plantvillage-dataset"

In [ ]:
import kagglehub

os.environ['KAGGLE_USERNAME'] = "<your_kaggle_username>"
os.environ['KAGGLE_KEY'] = "<your_kaggle_key>"

In [ ]:
temp_path = kagglehub.dataset_download(DATASET_NAME)

shutil.make_archive("/content/plantvillage_data", 'zip', temp_path)

shutil.move("/content/plantvillage_data.zip", os.path.join(BASE_PATH, "plant_data.zip"))

In [ ]:
shutil.unpack_archive(os.path.join(BASE_PATH, "plant_data.zip"), "/content/plantvillage_dataset")

In [ ]:
data_dir = os.path.join("/content/plantvillage_dataset", "plantvillage dataset", DATA_SUBDIR)

Import required libraries and optimise GPU

In [ ]:
import json, time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

for gpu in tf.config.list_physical_devices('GPU'):
  tf.config.experimental.set_memory_growth(gpu, True)

print(tf.__version__)
print(len(tf.config.list_physical_devices('GPU')))

Preparing Train and Test datasets

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split = 0.2, subset = "training",
    seed = SEED, image_size = (IMG_HEIGHT, IMG_WIDTH), batch_size = BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split = 0.2, subset = "validation",
    seed = SEED, image_size = (IMG_HEIGHT, IMG_WIDTH), batch_size = BATCH_SIZE
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

with open(LABELS_SAVE, "w") as f:
  json.dump(CLASS_NAMES, f, indent = 2)

Performing augmentation, rescaling and autotune operations (preprocessing)

In [ ]:
augmentation = Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
], name = "augmentation")

rescale = layers.Rescaling(1.0 / 255)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (train_ds.map(lambda x, y: (rescale(augmentation(x, training = True)), y), num_parallel_calls = AUTOTUNE).prefetch(AUTOTUNE))
val_ds = (val_ds.map(lambda x, y: (rescale(x), y), num_parallel_calls = AUTOTUNE).prefetch(AUTOTUNE))

Baseline CNN

In [ ]:
def build_cnn(input_shape, num_classes):
  inp = keras.Input(shape = input_shape)
  x = inp
  for filters in [32, 64, 128, 256]:
    x = layers.Conv2D(filters, 3, use_bias = False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(256, use_bias = False)(x)
  x = layers.BatchNormalization()(x)
  x = layers.Activation('relu')(x)
  x = layers.Dropout(0.5)(x)
  out = layers.Dense(num_classes, activation = "softmax")(x)
  return keras.Model(inp, out, name = "ReLeaf_Baseline_CNN")

model = build_cnn((IMG_HEIGHT, IMG_WIDTH, 3), NUM_CLASSES)
model.summary()
with open(BASE_PATH + "model_summary.txt", "w") as f:
  model.summary(print_fn = lambda x: f.write(x + "\n"))
model.compile(
    optimizer = keras.optimizers.Adam(1e-3),
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

Model Training

In [ ]:
callbacks = [
    ModelCheckpoint(MODEL_SAVE, monitor = "val_accuracy", save_best_only = True, mode = "max", verbose = 1),
    EarlyStopping(monitor = "val_loss", patience = 5, restore_best_weights = True, verbose = 1),
    ReduceLROnPlateau(monitor = "val_loss", factor = 0.5, patience = 3, min_lr = 1e-6, verbose = 1),
]

t0 = time.time()

history = model.fit(train_ds, validation_data = val_ds, epochs = NUM_EPOCHS, callbacks = callbacks)

print(f"Time to train: {((time.time() - t0) / 60):.2f} min")

Clear Plantvillage. To clear it type it in a cell and execute.

```bash
import subprocess

if os.path.exists(BASE_PATH):
    print(f"Attempting to force clear {BASE_PATH}...")

    try:
        subprocess.run(['rm', '-rf', BASE_PATH], check=True)
        print(f"Successfully cleared {BASE_PATH}")
    except subprocess.CalledProcessError as e:
        print(f"Error clearing {BASE_PATH}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred while clearing {BASE_PATH}: {e}")

os.makedirs(BASE_PATH, exist_ok=True)
```


Plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize = (14, 5))
fig.suptitle("ReLeaf Training History", fontweight = "bold")
for ax, metric in zip(axes, ["loss", "accuracy"]):
  ax.plot(history.history[metric], label = "Train")
  ax.plot(history.history[f"val_{metric}"], label = "Val")
  ax.set_title(metric.capitalize())
  ax.legend()
  ax.grid(alpha = 0.3)
plt.tight_layout()
plt.savefig(BASE_PATH + "training_history.png", dpi = 100)